# Prediction Landscape Analysis

The logit distribution landscape: statistical profiles,
decision margins, entropy, and probability concentration.

## Why This Matters

Prediction landscape analyzes how the model arrives at its output predictions. Tracking prediction formation across layers reveals the computational process — when the model commits to an answer, what changes its mind, and how confident it is.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prediction_landscape_analysis import (
    logit_distribution_profile, decision_margin_analysis,
    prediction_entropy_profile, logit_concentration,
    prediction_landscape_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Logit Distribution Profile

Statistical properties of the output logit vector.

In [ ]:
result = logit_distribution_profile(model, tokens, position=-1)
print(f"  Top token: {result['top_token']} (logit={result['top_logit']:.4f})")
print(f"  Mean={result['mean']:.4f}, Std={result['std']:.4f}")
print(f"  Range=[{result['min']:.4f}, {result['max']:.4f}] ({result['range']:.4f})")
print(f"  Entropy={result['entropy']:.4f}")

## Decision Margin

How confident is the top prediction vs alternatives?

In [ ]:
result = decision_margin_analysis(model, tokens, position=-1, top_k=10)
flag = ' [CONFIDENT]' if result['is_confident'] else ' [AMBIGUOUS]'
print(f"  Margin (1-2): {result['margin_1_2']:.4f}{flag}")
print(f"  Margin (1-3): {result['margin_1_3']:.4f}")
print(f"  Top predictions:")
for tok, logit in zip(result['top_tokens'], result['top_logits']):
    print(f"    Token {tok}: {logit:.4f}")

## Entropy Profile

Which positions are easy (low entropy) vs hard (high entropy)?

In [ ]:
result = prediction_entropy_profile(model, tokens)
print(f"  Mean entropy: {result['mean_entropy']:.4f}")
print(f"  Easiest position: {result['min_entropy_pos']}")
print(f"  Hardest position: {result['max_entropy_pos']}")
for p in result['per_position']:
    bar = '█' * int(p['entropy'])
    print(f"  Pos {p['position']}: H={p['entropy']:.4f}, top_p={p['top_prob']:.4f} {bar}")

## Logit Concentration

How much probability mass is in the top tokens?

In [ ]:
result = logit_concentration(model, tokens, position=-1)
flag = ' [CONCENTRATED]' if result['is_concentrated'] else ''
print(f"  Top-1 prob: {result['top_1_prob']:.4f}{flag}")
print(f"  Top-5 prob: {result['top_5_prob']:.4f}")
print(f"  Top-10 prob: {result['top_10_prob']:.4f}")
print(f"  Effective tokens: {result['effective_tokens']:.1f}")

## Prediction Landscape Summary

In [ ]:
result = prediction_landscape_summary(model, tokens, position=-1)
for k, v in result.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")